# Tech-Letter 본문 추출 & 청킹 전략 비교

이 노트북은 Tech-Letter 포스트들을 대상으로 다음을 실험합니다.

1. API로 포스트 메타데이터 조회
2. 세 가지 본문 추출 전략 비교
   - A: `aggregate_text` (Selenium + Html2Text)
   - B: `aggregate_text_dom_clean` (DOM 노이즈 제거 후 Html2Text)
   - C: `aggregate_text_trafilatura` (trafilatura ML 기반 본문 추출)
3. 제목 기반 크롭 + 라인 휴리스틱 필터 적용 후 LangChain 청킹
4. 전략별 텍스트 길이와 청킹 결과 개수를 비교

이 노트북은 **실험/튜닝용**이며, 프로덕션 파이프라인을 바꾸기 전에 아이디어를 검증하는 목적입니다.

In [6]:
from typing import Dict
from pathlib import Path

from langchain_text_splitters import RecursiveCharacterTextSplitter

from utils.aggregate import (
    aggregate_text,
    aggregate_text_dom_clean,
    aggregate_text_trafilatura,
)
from utils.api_client import TechLetterClient
from utils.data_loader import fetch_all_posts
from utils.doc_builders import build_document_from_full_text
from utils.app_config import CONFIG
from utils.content_filters import filter_by_title_crop, filter_by_line_heuristics

client = TechLetterClient(base_url=CONFIG.techletter_base_url)
posts = fetch_all_posts(client)

print(f"총 포스트 수: {len(posts)}")
for idx, post in enumerate(posts[:10], 1):
    print(f"[{idx}] {post.title} {post.id} {post.link}")

splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=150)

총 포스트 수: 189
[1] API 호출식 웜업의 부작용을 넘어서 : 라이브러리만 데우는 JVM 웜업 691d6d4b1529d465be9b2f05 https://d2.naver.com/helloworld/1580651
[2] Telegraf로 커스텀 지표 수집하기: Exporter 개발 경험 공유 691c0dbb1529d465be9b2f04 https://d2.naver.com/helloworld/8677833
[3] 6개월 만에 연간 수십조를 처리하는 DB CDC 복제 도구 무중단/무장애 교체하기 691be38b1529d465be9b2f03 https://d2.naver.com/helloworld/6388660
[4] 처음 만나는 OpenTelemetry (feat. Collector) 691ad85b1529d465be9b2f02 https://d2.naver.com/helloworld/1104856
[5] if(kakao)25 정규돈 CTO 키노트 후기 691aca4a1529d465be9b2f01 https://tech.kakao.com/posts/791
[6] 도장 파는 개발자 vs 공장 짓는 개발자 691a4bbe1529d465be9b2f00 https://medium.com/musinsa-tech/%EB%8F%84%EC%9E%A5-%ED%8C%8C%EB%8A%94-%EA%B0%9C%EB%B0%9C%EC%9E%90-vs-%EA%B3%B5%EC%9E%A5-%EC%A7%93%EB%8A%94-%EA%B0%9C%EB%B0%9C%EC%9E%90-b33dddf5daef?source=rss----f107b03c406e---4
[7] [무물보] 응답하라, 백엔드 개발자를 꿈꾸는 2025 학생 개발자! 6916c7bb1529d465be9b2efe https://d2.naver.com/news/5079232
[8] 100년 가는 프론트엔드 코드, SDK 6916c7cd1529d465be9b2eff http

In [2]:
# 본문 추출 + 필터 + 청킹 파이프라인 헬퍼

STRATEGIES = {
    "A_aggregate_text": aggregate_text,
    "B_dom_clean": aggregate_text_dom_clean,
    "C_trafilatura": aggregate_text_trafilatura,
}


def run_pipeline(post, extractor) -> Dict[str, object]:
    """단일 포스트에 대해 하나의 추출 전략을 적용하고 노이즈 제거 단계를 포함한 정보를 반환한다."""
    text_raw = extractor(post.link)
    text_title_crop = filter_by_title_crop(text_raw, title=post.title)
    text_line_filtered = filter_by_line_heuristics(text_title_crop)

    doc = build_document_from_full_text(post, text_line_filtered)
    chunks = splitter.split_documents([doc])

    return {
        "text_raw": text_raw,
        "text_title_crop": text_title_crop,
        "text_line_filtered": text_line_filtered,
        "raw_length": len(text_raw),
        "title_crop_length": len(text_title_crop),
        "line_filtered_length": len(text_line_filtered),
        "chunk_count": len(chunks),
        "chunks": chunks,
    }

In [3]:
# 단일 포스트에 대한 전략별 노이즈 제거 요약 + diff 보기

import difflib

# 필요 시 인덱스를 바꿔가며 테스트
TARGET_INDEX = 0


def _summary_stats(text: str) -> Dict[str, int]:
    lines = text.splitlines()
    return {"chars": len(text), "lines": len(lines)}


def _print_length_summary(post, results_by_strategy: Dict[str, Dict[str, object]]) -> None:
    print(f"타겟 포스트: {post.title}")
    print(f"링크: {post.link}\n")

    for name, res in results_by_strategy.items():
        raw_stats = _summary_stats(res["text_raw"])
        final_stats = _summary_stats(res["text_line_filtered"])

        removed_chars = raw_stats["chars"] - final_stats["chars"]
        removed_lines = raw_stats["lines"] - final_stats["lines"]
        removed_ratio = (
            removed_chars / raw_stats["chars"] * 100 if raw_stats["chars"] else 0.0
        )

        print(
            f"- {name}: chars {raw_stats['chars']} -> {final_stats['chars']} "
            f"({removed_chars} 제거, {removed_ratio:.1f}%), "
            f"lines {raw_stats['lines']} -> {final_stats['lines']} "
            f"({removed_lines} 제거), chunks={res['chunk_count']}"
        )


def _print_diff(original: str, cleaned: str, *, max_lines: int = 120, from_label: str, to_label: str) -> None:
    diff_iter = difflib.unified_diff(
        original.splitlines(),
        cleaned.splitlines(),
        fromfile=from_label,
        tofile=to_label,
        lineterm="",
        n=3,
    )
    diff_lines = list(diff_iter)

    if not diff_lines:
        print("변경 없음 (diff 비어 있음)")
        return

    if len(diff_lines) > max_lines:
        truncated = diff_lines[:max_lines]
        truncated.append(f"... ({len(diff_lines) - max_lines} lines truncated)")
        diff_lines = truncated

    for line in diff_lines:
        print(line)


# 실행 영역

target_post = posts[TARGET_INDEX]
results_by_strategy: Dict[str, Dict[str, object]] = {}

for name, extractor in STRATEGIES.items():
    results_by_strategy[name] = run_pipeline(target_post, extractor)

_print_length_summary(target_post, results_by_strategy)

for name, res in results_by_strategy.items():
    print(f"\n--- {name}: raw -> line_filtered diff ---")
    _print_diff(
        res["text_raw"],
        res["text_line_filtered"],
        from_label=f"{name}_raw",
        to_label=f"{name}_line_filtered",
        max_lines=120,
    )

타겟 포스트: API 호출식 웜업의 부작용을 넘어서 : 라이브러리만 데우는 JVM 웜업
링크: https://d2.naver.com/helloworld/1580651

- A_aggregate_text: chars 1177 -> 876 (301 제거, 25.6%), lines 106 -> 30 (76 제거), chunks=1
- B_dom_clean: chars 888 -> 716 (172 제거, 19.4%), lines 75 -> 24 (51 제거), chunks=1
- C_trafilatura: chars 709 -> 702 (7 제거, 1.0%), lines 31 -> 29 (2 제거), chunks=1

--- A_aggregate_text: raw -> line_filtered diff ---
--- A_aggregate_text_raw
+++ A_aggregate_text_line_filtered
@@ -1,106 +1,30 @@
-# naver D2
-
-메뉴
-
-  * Hello world
-  * D2 News
-  * About D2
-
-검색
-
- __
-
-검색 __
-
-# API 호출식 웜업의 부작용을 넘어서 : 라이브러리만 데우는 JVM 웜업
-
+API 호출식 웜업의 부작용을 넘어서 : 라이브러리만 데우는 JVM 웜업
  _등록일_
-
      2025.11.20
-|
-
-    |
  _코멘트_
-
-     285
-
 네이버 사내 기술 교류 행사인 NAVER ENGINEERING DAY 2025(10월)에서 발표되었던 세션을 공개합니다.  
-
-  
-
-#### 발표 내용
-
 API 호출식 웜업의 부작용을 개선한 라이브러리 웜업을 소개합니다.
-
-#### 발표 대상
-
 JVM JIT Compiler의 웜업 방식 기본을 알고 있는 분 또는 관심있는 분  
 JVM 기반 웹 어플리케이션의 웜업에 관심있는 분
-
-#### 목차
-
   * JVM WARM-UP?
   * 기존 웜업 방식과 문제
   * 아이디어
 

In [4]:
# 여러 포스트(상위 N개)에 대한 전략별 요약 비교

N = 10  # 필요 시 10 등으로 조정

for idx, post in enumerate(posts[:N], 1):
    print(f"\n=== [{idx}] {post.title} ===")

    for name, extractor in STRATEGIES.items():
        result = run_pipeline(post, extractor)
        print(
            f"- {name}: raw={result['raw_length']}, "
            f"라인필터={result['line_filtered_length']}, "
            f"chunks={result['chunk_count']}"
        )


=== [1] API 호출식 웜업의 부작용을 넘어서 : 라이브러리만 데우는 JVM 웜업 ===
- A_aggregate_text: raw=1177, 라인필터=876, chunks=1
- B_dom_clean: raw=888, 라인필터=716, chunks=1
- C_trafilatura: raw=709, 라인필터=702, chunks=1

=== [2] Telegraf로 커스텀 지표 수집하기: Exporter 개발 경험 공유 ===
- A_aggregate_text: raw=1367, 라인필터=1052, chunks=1
- B_dom_clean: raw=1072, 라인필터=886, chunks=1
- C_trafilatura: raw=722, 라인필터=711, chunks=1

=== [3] 6개월 만에 연간 수십조를 처리하는 DB CDC 복제 도구 무중단/무장애 교체하기 ===
- A_aggregate_text: raw=23971, 라인필터=22633, chunks=17
- B_dom_clean: raw=22864, 라인필터=21631, chunks=16
- C_trafilatura: raw=21308, 라인필터=21055, chunks=16

=== [4] 처음 만나는 OpenTelemetry (feat. Collector) ===
- A_aggregate_text: raw=1504, 라인필터=1241, chunks=1
- B_dom_clean: raw=1215, 라인필터=1081, chunks=1
- C_trafilatura: raw=1039, 라인필터=1032, chunks=1

=== [5] if(kakao)25 정규돈 CTO 키노트 후기 ===
- A_aggregate_text: raw=5119, 라인필터=4757, chunks=4
- B_dom_clean: raw=4261, 라인필터=4067, chunks=3
- C_trafilatura: raw=3635, 라인필터=3611, chu

In [5]:
# 여러 포스트의 전략별 최종 본문 텍스트를 outputs 디렉토리에 저장

from pathlib import Path

BASE_OUTPUT_DIR = Path("outputs/text")
BASE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

N_SAVE = 10  # 저장할 포스트 개수 (필요시 조정)

for idx, post in enumerate(posts[:N_SAVE], 1):
    print(f"\n=== 저장 대상 포스트 [{idx}] {post.title} ===")

    # post_id 기준 하위 폴더 하나에 전략별 결과를 모은다
    post_dir = BASE_OUTPUT_DIR / str(post.id)
    post_dir.mkdir(parents=True, exist_ok=True)

    for strategy_name, extractor in STRATEGIES.items():
        result = run_pipeline(post, extractor)
        text = result["text_line_filtered"]

        out_path = post_dir / f"{strategy_name}.txt"
        with out_path.open("w", encoding="utf-8") as f:
            f.write(text)

        print(
            f"  - {strategy_name}: 저장 완료 -> {out_path} "
            f"(chars={len(text)}, lines={len(text.splitlines())})"
        )


=== 저장 대상 포스트 [1] API 호출식 웜업의 부작용을 넘어서 : 라이브러리만 데우는 JVM 웜업 ===
  - A_aggregate_text: 저장 완료 -> outputs/text/691d6d4b1529d465be9b2f05/A_aggregate_text.txt (chars=876, lines=30)
  - B_dom_clean: 저장 완료 -> outputs/text/691d6d4b1529d465be9b2f05/B_dom_clean.txt (chars=716, lines=24)
  - C_trafilatura: 저장 완료 -> outputs/text/691d6d4b1529d465be9b2f05/C_trafilatura.txt (chars=702, lines=29)

=== 저장 대상 포스트 [2] Telegraf로 커스텀 지표 수집하기: Exporter 개발 경험 공유 ===
  - A_aggregate_text: 저장 완료 -> outputs/text/691c0dbb1529d465be9b2f04/A_aggregate_text.txt (chars=1046, lines=33)
  - B_dom_clean: 저장 완료 -> outputs/text/691c0dbb1529d465be9b2f04/B_dom_clean.txt (chars=886, lines=27)
  - C_trafilatura: 저장 완료 -> outputs/text/691c0dbb1529d465be9b2f04/C_trafilatura.txt (chars=711, lines=27)

=== 저장 대상 포스트 [3] 6개월 만에 연간 수십조를 처리하는 DB CDC 복제 도구 무중단/무장애 교체하기 ===
  - A_aggregate_text: 저장 완료 -> outputs/text/691be38b1529d465be9b2f03/A_aggregate_text.txt (chars=22633, lines=470)
  - B_dom_c